# Hand Gesture Recognition using OpenCV

## SkillCraft Technology - Machine Learning Internship

### Task 04

In [6]:
import cv2
import mediapipe as mp
import numpy as np
import time
from collections import deque

# MediaPipe Setup
mp_hands = mp.solutions.hands

# Finger Landmark Indices
FINGER_TIPS = [4, 8, 12, 16, 20]
FINGER_PIPS = [2, 6, 10, 14, 18]

# Colors
COL_GREEN = (0, 255, 0)
COL_BLUE = (255, 0, 0)
COL_RED = (0, 0, 255)
COL_WHITE = (255, 255, 255)
COL_BLACK = (0, 0, 0)

# Gesture Smoother
class GestureSmoother:

    def __init__(self, window=5):
        self.history = deque(maxlen=window)

    def update(self, gesture):
        self.history.append(gesture)
        return max(set(self.history), key=self.history.count)

# Finger State Detection
def get_finger_states(landmarks, hand_label):

    states = []

    # Thumb
    if hand_label == "Right":
        states.append(landmarks[4].x < landmarks[3].x)
    else:
        states.append(landmarks[4].x > landmarks[3].x)

    # Other Fingers
    for tip, pip in zip(FINGER_TIPS[1:], FINGER_PIPS[1:]):
        states.append(landmarks[tip].y < landmarks[pip].y)

    return states

# Gesture Classification
def classify_gesture(states):

    thumb, index, middle, ring, pinky = states
    count = sum(states)

    if count == 0:
        return "✊ Fist"

    if count == 5:
        return "🖐 Open Hand"

    if thumb and not index and not middle and not ring and not pinky:
        return "👍 Thumbs Up"

    if not thumb and index and middle and not ring and not pinky:
        return "✌️ Peace"

    if not thumb and index and not middle and not ring and not pinky:
        return "☝️ Pointing"

    if thumb and not index and not middle and not ring and pinky:
        return "🤙 Call Me"

    return f"{count} Fingers"

# Glassmorphism Panel
def draw_panel(img, x, y, w, h, color):

    overlay = img.copy()

    cv2.rectangle(
        overlay,
        (x, y),
        (x + w, y + h),
        color,
        -1
    )

    cv2.addWeighted(
        overlay,
        0.5,
        img,
        0.5,
        0,
        img
    )

# Webcam
cap = cv2.VideoCapture(0)

cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

smoother = GestureSmoother()

prev_time = time.time()

with mp_hands.Hands(
    model_complexity=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.6,
    max_num_hands=2
) as hands:

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame = cv2.flip(frame, 1)

        h, w = frame.shape[:2]

        # FPS
        current_time = time.time()

        fps = 1 / (current_time - prev_time)

        prev_time = current_time

        # RGB Conversion
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        results = hands.process(rgb)

        total_fingers = 0

        hand_gestures = []

        # Hand Detection
        if results.multi_hand_landmarks:

            for hand_landmarks, handedness in zip(
                results.multi_hand_landmarks,
                results.multi_handedness
            ):

                hand_label = handedness.classification[0].label

                landmarks = hand_landmarks.landmark

                # Draw Landmarks
                for idx, lm in enumerate(landmarks):

                    cx = int(lm.x * w)
                    cy = int(lm.y * h)

                    if idx in FINGER_TIPS:

                        cv2.circle(
                            frame,
                            (cx, cy),
                            8,
                            COL_GREEN,
                            -1
                        )

                    else:

                        cv2.circle(
                            frame,
                            (cx, cy),
                            5,
                            COL_BLUE,
                            -1
                        )

                # Draw Connections
                for connection in mp_hands.HAND_CONNECTIONS:

                    start = connection[0]
                    end = connection[1]

                    x1 = int(landmarks[start].x * w)
                    y1 = int(landmarks[start].y * h)

                    x2 = int(landmarks[end].x * w)
                    y2 = int(landmarks[end].y * h)

                    cv2.line(
                        frame,
                        (x1, y1),
                        (x2, y2),
                        COL_WHITE,
                        2
                    )

                # Finger Logic
                states = get_finger_states(
                    landmarks,
                    hand_label
                )

                hand_count = sum(states)

                total_fingers += hand_count

                gesture = classify_gesture(states)

                hand_gestures.append(
                    f"{hand_label}: {gesture}"
                )

        else:

            hand_gestures.append("No Hand Detected")

        # Gesture Text
        final_gesture = " | ".join(hand_gestures)

        final_gesture = smoother.update(final_gesture)

        # Top Info Panel
        draw_panel(
            frame,
            10,
            10,
            650,
            120,
            COL_BLACK
        )

        cv2.putText(
            frame,
            f"Gesture: {final_gesture}",
            (25, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            COL_GREEN,
            2
        )

        cv2.putText(
            frame,
            f"Total Fingers: {total_fingers}",
            (25, 95),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            COL_WHITE,
            2
        )

        # FPS Display
        cv2.putText(
            frame,
            f"FPS: {int(fps)}",
            (w - 180, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            COL_GREEN,
            2
        )

        # Exit Instruction
        cv2.putText(
            frame,
            "Press Q to Exit",
            (w - 250, h - 20),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            COL_RED,
            2
        )

        # Window
        cv2.imshow(
            "Hand Gesture Recognition - SkillCraft Task 04",
            frame
        )

        key = cv2.waitKey(1) & 0xFF

        if key == ord('q'):
            break

cap.release()

cv2.destroyAllWindows()

In [7]:
mp_hands = mp.solutions.hands